# Advisor review package -- opx ML thermobarometer

**Audience:** Dr. Kanani K.M. Lee
**Author:** Ta Quang Nhan (cadet, USCGA)
**Date:** 2026-04-20
**Target venue:** JGR ML & Computation

## Package purpose

Consolidates the post-Phase-1 state of the opx ML thermobarometer project
into a single reviewable artifact. Every number below loads from a CSV or
JSON under `results/`; nothing is hard-coded. If a required file is missing
the notebook cells raise FileNotFoundError rather than silently fabricating.

## Sections

1. Headline table (4 opx combos)
2. 8-cell winner table (tuned + TabPFN)
3. Bias-correction scoreboard
4. opx-only P_kbar per-regime breakdown
5. TabPFN head-to-head (pre vs post vs tuned vs Putirka)
6. Core figures (6 PDFs)
7. SI figures (7 PDFs)
8. Pre-registration (verbatim)
9. Methods summary
10. Limitations
11. Provenance

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path('.').resolve().parent.parent  # deliverables/lee_package_20260420/ -> project root
RESULTS = ROOT / 'results'
FIGS = Path('./figures')

def load_csv(relpath: str) -> pd.DataFrame:
    p = ROOT / relpath
    if not p.exists():
        raise FileNotFoundError(f'required CSV missing: {p}')
    return pd.read_csv(p)

def load_json(relpath: str) -> dict:
    p = ROOT / relpath
    if not p.exists():
        raise FileNotFoundError(f'required JSON missing: {p}')
    return json.loads(p.read_text())

pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print('Setup complete; loader ready.')

## 1. Headline table

Aggregate ALL-regime pre-correction RMSE, best-correction RMSE, and winner
verdict for each of the 4 opx combinations. Source:
`results/preregistered_scorecard_postcorrection.csv`.

In [ ]:
sc = load_csv('results/preregistered_scorecard_postcorrection.csv')
hl = sc[(sc['track'].isin(['opx_liq','opx_only'])) & (sc['regime']=='ALL')].copy()
display_cols = ['track','target','v10_pre_rmse','v10_post_rmse','best_external_rmse',
                'tabpfn_rmse','tabpfn_post_rmse','tabpfn_correction_form','winner']
hl = hl[display_cols].rename(columns={
    'v10_pre_rmse': 'tuned pre',
    'v10_post_rmse': 'tuned post',
    'best_external_rmse': 'Putirka best',
    'tabpfn_rmse': 'TabPFN pre',
    'tabpfn_post_rmse': 'TabPFN post',
    'tabpfn_correction_form': 'TabPFN form',
})
display(hl)

## 2. 8-cell winner table

The 8-cell view: each of the 4 opx combinations x (tuned best winner, TabPFN).
Source: `results/bias_correction_shipped.csv`.

In [ ]:
shipped = load_csv('results/bias_correction_shipped.csv')
opx_rows = shipped[shipped['pipeline']=='opx'].copy()
display_cols = ['track','target','model','winner','ship_a','ship_b','n_seeds_done']
display(opx_rows[display_cols])

## 3. Bias-correction scoreboard

TabPFN per-combo summary over 5 OOF seeds. Mean pre-correction RMSE, mean
Form A RMSE, mean Form B RMSE, ship counts, winner tallies. Source:
`results/tabpfn_bias_correction_summary.csv`.

In [ ]:
tf_sum = load_csv('results/tabpfn_bias_correction_summary.csv')
display(tf_sum)

Per-seed detail (5 seeds x 4 combos = 20 rows). Source:
`results/tabpfn_bias_correction_perseed.csv`.

In [ ]:
tf_per = load_csv('results/tabpfn_bias_correction_perseed.csv')
display(tf_per[['track','target','seed','pre_rmse_all','post_rmse_a','post_rmse_b','ship_a','ship_b','winner']])

## 4. opx-only P_kbar per-regime breakdown

Per-regime RMSE across the 5 candidates (tuned pre/post, Putirka 29c,
TabPFN pre/post). The post-corrected TabPFN is the aggregate winner on
3 of 5 regimes.

In [ ]:
pr = sc[(sc['track']=='opx_only') & (sc['target']=='P_kbar')].copy()
show = ['regime','n','v10_pre_rmse','v10_post_rmse','best_external_rmse',
        'tabpfn_rmse','tabpfn_post_rmse','tabpfn_correction_form','winner']
display(pr[show])

## 5. TabPFN head-to-head (aggregate RMSE, 20-seed baseline)

Source: `results/tabpfn_head_to_head.csv`. TabPFN vs best tuned family per
(pipeline, track, target) at aggregate ALL level from the 20-seed multiseed
protocol.

In [ ]:
h2h = load_csv('results/tabpfn_head_to_head.csv')
display(h2h)

## 6. Core figures

Six core manuscript figures. PDFs live in this package under `figures/`;
previews are PNG for rendering.

### fig44_tabpfn_bias_scoreboard_opx

Fig. 44. TabPFN v2 bias-correction scoreboard on the 4 opx combinations. Panel A: mean test RMSE (over 5 seeds) pre-correction (gray) and after applying Form A (blue, regime-piecewise linear) or Form B (orange, piecewise sigmoid) fit on 5-seed x 10-fold OOF residuals. Annotations show winner verdict under the conservative ship-if-better rule at canonical seed 42 (overall delta > 1e-6 and no regime degradation). Form A ships on opx_only/T_C and opx_only/P_kbar with large pre-post gaps (TabPFN materially over-predicts deeper mantle residual bias uncorrected). Form B ships nothing on opx TabPFN, consistent with the 0/8 tuned-family pattern. Panel B: per-seed ship stability over seeds 42-46; cells color winner (gray = none, blue = A, orange = B) annotated with the verdict. Shipping is consistent at seed 42, 43, 45 on opx_only/T_C and at all 5 seeds on opx_only/P_kbar. Sources: results/tabpfn_bias_correction_perseed.csv, results/tabpfn_bias_correction_summary.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig44_tabpfn_bias_scoreboard_opx.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig45_opx_only_P_headline

Fig. 45. Per-regime RMSE on the opx-only P_kbar test set across five candidates: the tuned ML baseline tuned pre-correction (gray), the tuned ML baseline tuned post-correction (Form A, blue), Putirka 29c (red), TabPFN v2 pre-correction (purple), TabPFN v2 post-correction (Form A, green hatched). Regimes follow the pre-registered pressure partition (<5, 5-10, 10-20, >=20 kbar) plus ALL. Error bars are 95% bootstrap CIs (n_boot = 500). TabPFN post-correction wins 3 of the 5 regimes (lithospheric_mantle, deeper_mantle, ALL) and is within the the tuned ML baseline corrected CI on the remaining two. Form A transforms TabPFN from the worst pre-correction method into the aggregate winner, demonstrating that the in-context predictions carry a systematic deeper-regime bias that can be removed with a regime-piecewise linear transform. Source: results/preregistered_scorecard_postcorrection.csv (canonical seed 42 for TabPFN post).

In [ ]:
from IPython.display import Image
png = FIGS / 'fig45_opx_only_P_headline.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig34_bias_correction_scorecard_delta

Fig. 34. Scorecard delta between post-correction-correction RMSE and the best external baseline per pre-registered regime, split by pipeline (opx left, cpx right). Color encodes fractional improvement (best_external - post-correction)/best_external on a diverging scale: green = the tuned ML baseline wins, red = external wins, white = tie. Cells annotated with the absolute RMSE delta in native units. Cells without an external baseline list the absolute post-correction value. External reference methods include Putirka (2008) thermobarometers, Agreda-Lopez (2024) ML cpx models, and Jorgenson (2022) ML cpx models. Source CSV: results/preregistered_scorecard_postcorrection.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig34_bias_correction_scorecard_delta.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig35_tabpfn_vs_opx_tb

Fig. 35. TabPFN v2 (Hollmann et al. 2025) versus the pre-correction-registered tuned baseline and the best classical/ML external reference per (pipeline, track, target) combination. TabPFN error bars show 5-seed ensemble stability; the tuned ML baseline error bars show 20-seed model-fit variance. TabPFN receives raw oxide features only (no ALR/PWLR) and is fit with default hyperparameters (n_estimators=8 opx / 4 cpx, device=cpu). Sources: results/tabpfn_multiseed_summary.csv, results/tabpfn_head_to_head.csv, results/v10_{opx,cpx}_multiseed_summary.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig35_tabpfn_vs_opx_tb.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

        ### fig28_bias_correction_opx_liq

        Fig. 28. Per-regime piecewise linear bias correction for the opx-liq
canonical cells. Correction coefficients (y_corrected = a * y_pred + b)
are fit on train-set out-of-fold predictions (StratifiedGroupKFold 10
folds, Citation-grouped, target stratified by NB03 bins) and applied to
the held-out test set per pre-registered P regime (shallow_crustal
[0,5) kbar; deep_crustal_MASH [5,15) kbar; lithospheric_mantle [15,30) kbar;
deeper_mantle [30,100) kbar). Panels show test-set RMSE before vs after
correction per regime plus the pooled "ALL" row. Result: P_kbar
corrections improve ALL RMSE (MLP 3.82 to 2.58 kbar; ElasticNet 5.59
to 3.05 kbar). T_C correction is a neutral-to-negative wash on
ElasticNet/raw (pooled 77.06 to 77.17; shallow_crustal HURT 35.33 to
54.01), consistent with the v9 finding that train-OOF T bias does not
transfer to the held-out test set. Produced by
scripts/v10_phase_g_nb07_bias.py. Supports T09 (bias correction
effectiveness). Source CSVs: results/opx_liq_bias_correction.csv,
results/opx_liq_bias_correction_params.csv.

In [ ]:
from IPython.display import Image
png = FIGS / 'fig28_bias_correction_opx_liq.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig_nb08_twopx_1to1

(no caption sidecar)

In [ ]:
from IPython.display import Image
png = FIGS / 'fig_nb08_twopx_1to1.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

## 7. Supporting information figures

### fig30_bias_correction_per_regime_rmse

Fig. 30. Per-regime test-set RMSE before and after bias correction for the 8 Phase-G.7 cells. Grid rows: T_C (top), P_kbar (bottom). Columns: opx-liq, opx-only, cpx-liq, cpx-only. Each regime group shows pre-correction (gray), post Form A (blue), post Form B (vermillion). Bar heights are 20-seed means; vertical error bars are the mean of per-seed bootstrap 95% CIs. Hatched bars mark regimes with n < 20 (sample-size limited). "ships A/B" annotation marks the Phase-G.7 shipping decision at canonical seed 42. Source CSVs: results/bias_correction_{per_seed,summary,shipped}.csv.

In [ ]:
png = FIGS / 'fig30_bias_correction_per_regime_rmse.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig31_bias_correction_residuals

Fig. 31. Residual-vs-predicted scatter for the 8 Phase-G.7 cells at canonical seed 42. Gray markers: pre-correction residuals. Colored markers: residuals after the shipping form of correction, grouped by each sample pre-registered P regime (shallow_crustal blue, deep_crustal_MASH orange, lithospheric_mantle green, deeper_mantle vermillion). A shaded band marks +/-1 sigma of the post-correction residuals. Cells whose shipping decision is "none" show the pre-correction residuals alone and are labelled accordingly. Per-sample predictions loaded from results/bias_correction/checkpoints/ (seed 42). Source CSV: results/bias_correction_shipped.csv.

In [ ]:
png = FIGS / 'fig31_bias_correction_residuals.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig32_bias_correction_form_comparison

Fig. 32. Test-set RMSE at regime=ALL comparing pre-correction with Form A and Form B post-correction for the 8 Phase-G.7 cells (20-seed mean). Shipped form (if any) is drawn with a bold border. Panel subtitle reports the Phase-G.7 shipping decision and the per-seed vote split across 20 model seeds. Source CSVs: results/bias_correction_{summary,per_seed,shipped}.csv.

In [ ]:
png = FIGS / 'fig32_bias_correction_form_comparison.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

### fig33_bias_correction_per_seed_stability

Fig. 33. Per-seed $\Delta$RMSE (pre minus post, positive = improvement) at regime=ALL for Form A (x-axis) vs Form B (y-axis), one point per model seed across the 20-seed protocol. Points are colored by the per-seed winner (blue = A, vermillion = B, gray = none). Light-green shading highlights the upper-right quadrant where both forms improve test RMSE; light-red shading highlights the lower-left quadrant where both forms degrade it. Panel annotation reports the per-seed vote count. Source CSV: results/bias_correction_per_seed.csv.

In [ ]:
png = FIGS / 'fig33_bias_correction_per_seed_stability.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

        ### fig_aug01_ship_verdict_comparison

        fig_aug01_ship_verdict_comparison

Ship-if-better verdict under the Agreda-Lopez (2024) 15x augmentation protocol versus the non-augmented baseline, across the four opx combinations (track x target). Bars show Form A / Form B / none ship counts across 20 seeds per condition.

In [ ]:
png = FIGS / 'fig_aug01_ship_verdict_comparison.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

        ### fig_aug02_aggregate_rmse_delta

        fig_aug02_aggregate_rmse_delta

Aggregate test RMSE under the non-augmented baseline versus the 15x augmented condition per opx combination. Error bars are 20-seed standard deviation. Putirka classical reference is overlaid where available.

In [ ]:
png = FIGS / 'fig_aug02_aggregate_rmse_delta.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

        ### fig_aug03_residual_structure_per_regime

        fig_aug03_residual_structure_per_regime

OOF residual distributions by pressure regime for opx-only P_kbar, non-augmented (top row) versus 15x augmented (bottom row). Violin plots show Form A and Form B fit substrate under each protocol.

In [ ]:
png = FIGS / 'fig_aug03_residual_structure_per_regime.png'
if png.exists():
    display(Image(filename=str(png), width=780))
else:
    display(Markdown(f'**missing:** {png}'))

## 8. Pre-registration (verbatim)

The pre-registered pressure partition and test protocol, reproduced
verbatim from `docs/preregistration/`.

In [ ]:
for name in ('p_regime_preregistration.md', 'nb03_test_protocol.md'):
    p = ROOT / 'docs' / 'preregistration' / name
    if not p.exists():
        raise FileNotFoundError(p)
    display(Markdown(f'### `{name}`'))
    display(Markdown(p.read_text(encoding='utf-8')))

## 9. Methods summary

- **Pipelines:** opx_liq (pyroxene + liquid features) and opx_only
  (pyroxene-only features). Four (track, target) combinations: opx_liq T_C,
  opx_liq P_kbar, opx_only T_C, opx_only P_kbar.
- **Training universe:** LEPR experimental database (ExPetDB 2025-07-21)
  filtered for opx equilibrium and citation-grouped into folds so that
  no citation appears in both a fold's train and held-out split.
- **Tuned families:** 8 gradient-boosted / forest / linear families
  (RF, ERT, XGB, GB, CatBoost, LightGBM, ElasticNet, MLP), each Optuna-
  tuned (50 trials, 12 inner jobs) on the 5-fold CV objective.
- **Foundation baseline:** TabPFN v2 (Hollmann et al. 2025) with
  `n_estimators=8` on CPU, no tuning. Bias-corrected variant uses
  5-seed 10-fold out-of-fold residuals as the correction-fit substrate.
- **Pressure partition:** shallow_crustal (<5 kbar), deep_crustal_MASH
  (5-10), lithospheric_mantle (10-20), deeper_mantle (>=20), plus ALL.
  Registered 2026-04-17 before any correction fitting.
- **Correction forms:**
    - *Form A:* per-regime ordinary least squares y_corr = a_r * y_pred + b_r.
    - *Form B:* piecewise sigmoid blend in Agreda-Lopez (2024) form with
      data-driven breakpoints.
- **Ship-if-better rule:** conservative acceptance -- Form ships iff
  `overall_delta > 1e-6` AND `max_regime_degradation <= 1e-6`. No
  per-regime loss accepted for aggregate gain.
- **Multiseed protocol:** 20 seeds (42-61) for test-RMSE variance; 5 seeds
  for OOF bias-correction fit (per standard protocol).

## 10. Limitations

1. **Form B fails to ship on opx.** 0/8 tuned + 0/4 TabPFN opx combinations
   pass the ship-if-better rule under Form B. The augmentation ablation
   (nb04b, 15x Gaussian noise at 3% RSD per Agreda-Lopez 2024) does NOT
   rescue Form B; aggregate RMSE strictly degrades under augmentation on
   every opx combination. Attribution: mineral-specific or dataset-size
   specific, not protocol-specific.
2. **opx-only T is a null result.** Neither Form A nor Form B ships on
   opx_only/T_C for any tuned family. TabPFN Form A ships but only on the
   deeper_mantle regime; the aggregate ALL improvement is marginal. The
   opx-only thermometer is not recommended for deployment.
3. **Natural-sample agreement is imperfect.** On LEPR paired pyroxenes
   (n=327), opx-only ML carries a +80 C positive T bias against both
   Jorgenson cpx-only and Putirka two-pyroxene methods. P agreement is
   within the conformal half-width.
4. **5-seed OOF for TabPFN is a reduction from the 20-seed test protocol.**
   The bias-fit substrate uses 5 seeds to bound CPU cost; test-set
   inference uses the full 20 seeds. TabPFN Form A ship decisions are at
   canonical seed 42 with per-seed stability reported.
5. **No external GEOROC cpx-opx pair update since Apr 9 2026.** The natural-
   sample refresh uses the existing on-disk export; the upstream server
   URL / credential were not supplied for this round.

## 11. Provenance

Git state + source CSV sizes at build time. Loaded via `PROVENANCE.md`
alongside this notebook.

In [ ]:
prov = Path('./PROVENANCE.md')
if prov.exists():
    display(Markdown(prov.read_text(encoding='utf-8')))
else:
    display(Markdown('(provenance file missing)'))